# Streamflow Data Download

Metadata for streamflow to be downloaded from: https://wateroffice.ec.gc.ca/station_metadata/station_characteristics_e.html using the specifications `Province = All Provinces/Territories`, `Parameter Type = Flows`, and `Regulation = Natural`. Save this file to `data/raw/station_metadata.csv`.

The stations listed in the metadata file subject to the date specifications are downloaded below from HYDAT and saved to `combined_streamflow.csv`.

In [ ]:
import pandas as pd
from src.data_ingestion import fetch_streamflow_batch
from src.processing import filter_stations_by_annual_completeness
from src.config import RAW_DATA_DIR, DATA_START_YEAR, DATA_END_YEAR, PROCESSED_DATA_DIR

METADATA_PATH = RAW_DATA_DIR / "station_metadata.csv"
OUTPUT_FILENAME = "combined_streamflow.csv"
TARGET_MDAS = ["05", "07", "08"]  # 05=Nelson River, 07=Great Slave Lake, 08=Pacific
TARGET_PROVS = ["BC", "AB"]

print("--- Filtering Metadata ---")
metadata = pd.read_csv(METADATA_PATH)

# Create Masks
# 1. Temporal Coverage
mask_time = (metadata['Year From'] <= DATA_START_YEAR) & (metadata['Year To'] >= DATA_END_YEAR)

# 2. Province (BC or AB)
mask_prov = metadata['Province'].isin(TARGET_PROVS)

# 3. Major Drainage Area (First 2 digits of ID)
# We assume 'Station Number' exists (e.g., '08GA010')
mask_mda = metadata['Station Number'].str[:2].isin(TARGET_MDAS)

# Apply Combined Filter
filtered_metadata = metadata[mask_time & mask_prov & mask_mda]
stations_to_download = filtered_metadata["Station Number"].unique().tolist()

print(f"Found {len(stations_to_download)} stations matching criteria in {TARGET_PROVS} (MDAs {TARGET_MDAS}).")

# --- Download & Process ---
if stations_to_download:
    print(f"\n--- Downloading Data ({DATA_START_YEAR}-{DATA_END_YEAR}) ---")
    
    # fetch_streamflow_batch returns the DataFrame
    df_raw = fetch_streamflow_batch(
        stations_to_download, 
        DATA_START_YEAR, 
        DATA_END_YEAR
    )
    
    print(f"   Raw Download Shape: {df_raw.shape[1]} stations")

    print("\n--- 3. Filtering by Completeness (<50% missing) ---")
    # Apply the completeness filter in-memory
    df_clean = filter_stations_by_annual_completeness(df_raw, max_missing_pct=50.0)
    
    print(f"   Cleaned Data Shape: {df_clean.shape[1]} stations")
    
    # Save the FINAL clean file
    save_path = PROCESSED_DATA_DIR / OUTPUT_FILENAME
    df_clean.to_csv(save_path)
    print(f"✅ Saved filtered, high-quality data to: {save_path}")
else:
    print("❌ No stations found matching criteria.")

--- Filtering Metadata ---
Found 360 stations matching criteria in ['BC', 'AB'] (MDAs ['05', '07', '08']).

--- Downloading Data (1980-2022) ---
15706 days of data saved for 360 stations
   Raw Download Shape: 360 stations

--- 3. Filtering by Completeness (<50% missing) ---
Filtering at 50.0% annual threshold:
 - Keeping 269 stations.
 - Dropping 91 stations due to incomplete years.
   Cleaned Data Shape: 269 stations
✅ Saved filtered, high-quality data to: D:\Work\MSc\Reproducibility\lstm_glacier_counterfactuals\data\processed\combined_streamflow.csv


## Monthy Glacier Mass Balance Reconstruction Data
Data was produced by Christina Draeger for the years 1980 to 2022 and can be accessed via: https://www.dropbox.com/scl/fo/yat0rxeoztpwol29qput2/AEtDmgySFbMEr3B9YcwLmks/kp_dp_alphabias_monthly_NN?dl=0&rlkey=4t3uobuuo8ufn5selgr5afoo4&subfolder_nav_tracking=1

Save the files under `data/raw/mass_balance`.

## Downloading Glacier Areas
Spatial information for the glaciers is downloaded from the [Randolph Glacier Inventory (RGI) version 6](https://daacdata.apps.nsidc.org/pub/DATASETS/nsidc0770_rgi_v6/). The region for Western Canada and US (`nsidc0770_02.rgi60.WesternCanadaUS.zip`) is the only download required. Save the unzipped files under `data/raw/RGI-western-canada`

## Downloading Drainage Areas
Water basin polygons can be downloaded from https://collaboration.cmc.ec.gc.ca/cmc/hydrometrics/www/HydrometricNetworkBasinPolygons/gpkg/. The major drainage areas (MDA) selected are:
* (5) Nelson River
* (7) Great Slave Lake
* (8) Pacific

These MDAs were selected due to their proximity to the Coast and Rocky Mountain Ranges. The files are saved under `data/raw/drainage_areas/`.

## Preprocessing Streamflow Data and Computing Basin Attributes
Streamflow data is filtered to remove any stations that have less that 60% of daily data available for any given year. Stations that are outside the selected MDAs are also removed.

The area, mean elevation, and percent glaciation of each remaining station is computed and saved to `data/processed/static_attributes.csv`. The monthly mass balance data was also used to compute the monthly change in glacier volume in million cubic meters for each glacierized basin. This data is saved in the following files:
* `data/raw/mass_balance/ts_monthly_const_area_lstm.csv` data are saved to `data/processed/glacier_volume_change_1.csv`.
* `data/raw/mass_balance/ts_monthly_const_area_fnn.csv` data are saved to `data/processed/glacier_volume_change_2.csv`.
* `data/raw/mass_balance/ts_monthly_const_area_fnn_cluster.csv` data are saved to `data/processed/glacier_volume_change_3.csv`.

In [1]:
import pandas as pd
from src.data_ingestion import download_aws_dem
from src.processing import compute_and_save_bounds
from src.spatial_utils import process_spatial_attributes
from src.config import PROCESSED_DATA_DIR

# 1. Load the streamflow data
print("--- Loading Streamflow ---")
df_flow = pd.read_csv(PROCESSED_DATA_DIR / "combined_streamflow.csv", index_col="Date", parse_dates=True)
stations = df_flow.columns.tolist()

print(f"Loaded clean data: {df_flow.shape[1]} stations.")

# 2. Compute Bounds (Step 1)
bounds = compute_and_save_bounds(stations)

# 3. Download DEM (Step 2)
# This will save to data/raw/dem_data/western_canada_dem.tif
download_aws_dem(bounds) 

# 4. Process Attributes (Step 3)
# This uses the DEM we just downloaded
static_df, vol_df = process_spatial_attributes(stations)

--- Loading Streamflow ---
Loaded clean data: 269 stations.
⏳ Computing spatial bounds from basin files...
✅ Spatial bounds saved to D:\Work\MSc\Reproducibility\lstm_glacier_counterfactuals\data\raw\spatial_bounds.csv
   Bounds: {'north': 61, 'south': 48, 'east': -106, 'west': -133}
ℹ️ DEM already exists at D:\Work\MSc\Reproducibility\lstm_glacier_counterfactuals\data\raw\dem_data\western_canada_dem.tif. Skipping download.
⏳ Loading and merging basin files...
✅ Processing 269 basins.
⏳ Extracting Elevation and Topography Basin-by-Basin (Zero Disk Storage)...
⏳ Merging Station Coordinates...
⏳ Intersecting Glaciers and Calculating Distances...
✅ Static attributes saved to D:\Work\MSc\Reproducibility\lstm_glacier_counterfactuals\data\processed\static_attributes.csv
⏳ Validating Mass Balance Coverage and Calculating Volume Changes...
   ⚠️ Excluding 8 basins from volume calculations due to missing mass balance data (e.g., US headwaters).
   -> Processing Member 1: ts_monthly_const_area_ls

## Downloading ERA5 Climate Data
This project uses the Copernicus Climate Data Store (CDS) to download ERA5 precipitation and temperature data. Follow these steps to configure your environment.

#### Create a CDS Account

1. Visit the [Climate Data Store (CDS) registration page](https://cds.climate.copernicus.eu/#!/home).

2. Create an account and log in.

#### Accept the Terms of Use
Important: You must manually accept the "Terms of Use" for every dataset you wish to download, or the API will return an error.

1. Go to the ERA5 daily statistics page.

2. Click the "Download Data" tab.

3. Scroll to the bottom and click Accept Terms (look for a "License" section).

4. Repeat this for the ERA5 reanalysis single levels.

#### Get your API Key
1. Go to your User Profile.

2. Scroll down to the section labeled API Key.

3. You will see a block of text that looks like this:

```
url: https://cds.climate.copernicus.eu/api/v2
key: <UID>:<API-KEY>
```
#### Configure the Configuration File (`.cdsapirc`)
The cdsapi library looks for a hidden file in your home directory to authenticate.

**For Windows Users:**

1. Open your User folder (e.g., C:\Users\YourName).

2. Create a new text file named .cdsapirc (Note the leading dot).

* Tip: If Windows doesn't let you create a file starting with a dot, name it .cdsapirc. (with a dot at the end) and it will save correctly.

3. Paste the url and key from Step 3 into this file.

**For Mac/Linux Users:**

1. Open your terminal.

2. Run the following command: `nano ~/.cdsapirc`

3. Paste your credentials:
```
url: https://cds.climate.copernicus.eu/api/v2
key: 12345:abcdefgh-ijkl-mnop-qrst-uvwxyz
```
4. Save and exit (`Ctrl+O`, `Enter`, `Ctrl+X`).

In [2]:
from src.data_ingestion import download_era5_precipitation, download_era5_temperature

# Study parameters
STUDY_YEARS = range(1979, 2023)

# Run downloads
download_era5_precipitation(STUDY_YEARS)
download_era5_temperature(STUDY_YEARS)

✔ Skipping era5_precip 1979-01 (already exists)
✔ Skipping era5_precip 1979-02 (already exists)
✔ Skipping era5_precip 1979-03 (already exists)
✔ Skipping era5_precip 1979-04 (already exists)
✔ Skipping era5_precip 1979-05 (already exists)
✔ Skipping era5_precip 1979-06 (already exists)
✔ Skipping era5_precip 1979-07 (already exists)
✔ Skipping era5_precip 1979-08 (already exists)
✔ Skipping era5_precip 1979-09 (already exists)
✔ Skipping era5_precip 1979-10 (already exists)
✔ Skipping era5_precip 1979-11 (already exists)
✔ Skipping era5_precip 1979-12 (already exists)
✔ Skipping era5_precip 1980-01 (already exists)
✔ Skipping era5_precip 1980-02 (already exists)
✔ Skipping era5_precip 1980-03 (already exists)
✔ Skipping era5_precip 1980-04 (already exists)
✔ Skipping era5_precip 1980-05 (already exists)
✔ Skipping era5_precip 1980-06 (already exists)
✔ Skipping era5_precip 1980-07 (already exists)
✔ Skipping era5_precip 1980-08 (already exists)
✔ Skipping era5_precip 1980-09 (already 

## Preprocess Climate Data
Compute daily basin averaged statistics for each climate vraible.

In [1]:
import pandas as pd
from src.climate import process_era5_basin_data
from src.config import DRAINAGE_FILES, PROCESSED_DATA_DIR

# flow_df contains the columns of the stations we want
flow_df = pd.read_csv(PROCESSED_DATA_DIR / "combined_streamflow.csv", index_col=0)
flow_df.columns

process_era5_basin_data(
    basin_gpkg_list=DRAINAGE_FILES,
    stations_list=flow_df.columns.tolist()
)

Step 1/5: Loading Basins...
   🔄 Reprojecting basins to EPSG:4326 (Lat/Lon)...
Step 2/5: Mapping Spatial Weights...
⏳ Computing spatial weights for 269 basins...


Mapping Grid: 100%|██████████| 269/269 [03:45<00:00,  1.19it/s]



Step 3/5: Processing Total Precipitation...


Precip Files: 100%|██████████| 528/528 [17:06<00:00,  1.94s/it]


✅ Total Precipitation data saved.

Step 4/5: Processing Temperature...


Temp Files: 100%|██████████| 528/528 [18:54<00:00,  2.15s/it]


✅ Temperature data saved.

Step 5/5: Splitting Precipitation into Rain/Snow and computing Freezing Fraction...


Snow/Rain/Frac Split: 100%|██████████| 528/528 [38:16<00:00,  4.35s/it]


✅ Snowfall, Rainfall, and Fraction Below Zero data saved.

🎉 Climate processing complete.
